In [3]:
!pip install gradio scikit-learn pandas numpy --quiet

import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import gradio as gr

# 1. Load Dataset & Train Model
print("Loading Medical Dataset...")
data = load_breast_cancer()

selected_features = [
    'mean radius',
    'mean texture',
    'mean perimeter',
    'mean area',
    'mean smoothness'
]
feature_indices = [list(data.feature_names).index(f) for f in selected_features]

X = data.data[:, feature_indices]
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

disease_model = RandomForestClassifier(n_estimators=100, random_state=42)
disease_model.fit(X_train, y_train)

# 2. Updated Prediction Function with Explicit Disease Labels
def predict_disease(radius, texture, perimeter, area, smoothness):
    input_data = np.array([[radius, texture, perimeter, area, smoothness]])

    prediction = disease_model.predict(input_data)[0]
    probabilities = disease_model.predict_proba(input_data)[0]

    return {
        "Healthy / Non-Cancerous (Benign Tumor)": float(probabilities[1]),
        "Breast Cancer Present (Malignant Tumor)": float(probabilities[0])
    }

# 3. Launch UI with Clear Title & Descriptions
inputs = [
    gr.Slider(6.0, 30.0, value=14.0, label="Mean Radius"),
    gr.Slider(9.0, 40.0, value=19.0, label="Mean Texture"),
    gr.Slider(43.0, 190.0, value=90.0, label="Mean Perimeter"),
    gr.Slider(140.0, 2500.0, value=650.0, label="Mean Area"),
    gr.Slider(0.05, 0.2, value=0.1, label="Mean Smoothness")
]

interface = gr.Interface(
    fn=predict_disease,
    inputs=inputs,
    outputs=gr.Label(num_top_classes=2),
    title="Breast Cancer Prediction System",
    description="Adjust patient diagnostic tumor metrics below to predict Breast Cancer risk in real-time."
)

interface.launch(share=True)

Loading Medical Dataset...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6b89ed35e8ec4239b9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
